# RECARD: Multi-Model Fraud Detection & Behavioral Profiling

**Authors**: Punyap Tejas Kumar & Shaswat Sahu  
**Project**: RECARD Multi-Model Fraud Intelligence  

This notebook compiles the end-to-end Machine Learning pipeline and visual comparative analytics:
1. **Exploratory Data Analysis & Dimensionality Reduction (PCA / t-SNE 2D & 3D Scatter Plots)**
2. **Comparative Feature Distribution & Correlation Heatmaps**
3. **Multi-Model Training**: 
   - **K-Nearest Neighbors (KNN)** (Distance/Density Outlier)
   - **Random Forest (RF)** (Balanced Ensemble Tree Partitioning)
   - **XGBoost (XGB)** (Gradient Boosting with Imbalance Cost-Weighting)
   - **Sequential Neural Network (LSTM / RNN)** for temporal burst & velocity profiling
4. **Multi-Model Comparative Visualizations**:
   - Precision-Recall (PR) Curves & AUPRC (suited for high imbalance)
   - ROC Curves & AUC Comparison
   - Confusion Matrices Comparison
   - **Model Prediction Confidence vs. Transaction Amount Scatter Plot**
   - **Anomaly Score vs. True Class Latent Space Scatter Comparison**

In [ ]:
# 1. Import necessary libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.metrics import (
    precision_recall_curve, auc, roc_curve, roc_auc_score,
    classification_report, confusion_matrix, f1_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All libraries loaded successfully.')

In [ ]:
# 2. Load Dataset (or generate standardized synthetic dataset if offline)
try:
    import kagglehub
    print('Attempting kagglehub dataset fetch...')
    ulb_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
    csv_file = [f for f in os.listdir(ulb_path) if f.endswith('.csv')][0]
    df = pd.read_csv(os.path.join(ulb_path, csv_file))
    print(f'Successfully loaded Credit Card Fraud dataset: {df.shape}')
except Exception as e:
    print(f'Kagglehub fetch skipped ({e}). Generating representative transaction dataset...')
    np.random.seed(42)
    n_normal = 20000
    n_fraud = 250
    
    # Normal transactions: centered around 0 with unit variance
    norm_pca = np.random.normal(loc=0.0, scale=1.0, size=(n_normal, 28))
    norm_time = np.sort(np.random.uniform(0, 172800, size=(n_normal, 1)))
    norm_amount = np.random.exponential(scale=70.0, size=(n_normal, 1)) + 1.0
    norm_data = np.hstack([norm_time, norm_pca, norm_amount, np.zeros((n_normal, 1))])
    
    # Fraud transactions: shifted distributions, extreme values, distinct PCA cluster
    fraud_pca = np.random.normal(loc=2.5, scale=2.0, size=(n_fraud, 28))
    fraud_pca[:, :4] -= 3.5  # shift prominent PCA components V1-V4
    fraud_time = np.random.uniform(0, 172800, size=(n_fraud, 1))
    fraud_amount = np.random.exponential(scale=450.0, size=(n_fraud, 1)) + 120.0
    fraud_data = np.hstack([fraud_time, fraud_pca, fraud_amount, np.ones((n_fraud, 1))])
    
    cols = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
    all_data = np.vstack([norm_data, fraud_data])
    df = pd.DataFrame(all_data, columns=cols).sort_values('Time').reset_index(drop=True)
    print(f'Representative dataset synthesized: {df.shape}')

print('Class balance:\n', df['Class'].value_counts())

In [ ]:
# 3. Exploratory Scatter Plot: Latent PCA Dimensions & Transaction Amount Comparison
plt.figure(figsize=(14, 5))

# Subplot 1: 2D Scatter Plot of Latent Features V1 vs V2 separated by Class
plt.subplot(1, 2, 1)
normal_mask = df['Class'] == 0
fraud_mask = df['Class'] == 1

plt.scatter(
    df.loc[normal_mask, 'V1'], df.loc[normal_mask, 'V2'],
    c='#2a9d8f', alpha=0.25, s=15, label='Legitimate Transaction (Class 0)', edgecolors='none'
)
plt.scatter(
    df.loc[fraud_mask, 'V1'], df.loc[fraud_mask, 'V2'],
    c='#e76f51', alpha=0.85, s=40, label='Fraudulent Transaction (Class 1)', edgecolors='black', linewidth=0.5
)
plt.title('Scatter Plot: V1 vs V2 Feature Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Principal Component V1')
plt.ylabel('Principal Component V2')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)

# Subplot 2: Scatter Plot of Transaction Amount vs V4 Dimension
plt.subplot(1, 2, 2)
plt.scatter(
    df.loc[normal_mask, 'V4'], df.loc[normal_mask, 'Amount'],
    c='#264653', alpha=0.25, s=15, label='Legitimate', edgecolors='none'
)
plt.scatter(
    df.loc[fraud_mask, 'V4'], df.loc[fraud_mask, 'Amount'],
    c='#d62828', alpha=0.9, s=45, label='Fraud (Anomalous Cluster)', edgecolors='yellow', linewidth=0.6
)
plt.yscale('log')
plt.title('Scatter Plot: V4 vs Log-Amount Anomaly Boundary', fontsize=12, fontweight='bold')
plt.xlabel('Component V4')
plt.ylabel('Transaction Amount (Log Scale $)')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# 4. Chronological Train-Test Split and Model Pipeline Setup
X = df.drop('Class', axis=1)
y = df['Class'].astype(int)

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Build Pipelines for KNN, RF, and XGBoost
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
])

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=100, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1
    ))
])

# Calculate scale_pos_weight for imbalance adjustment
pos_weight = max(1.0, float((y_train == 0).sum() / max(1, (y_train == 1).sum())))
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', xgb.XGBClassifier(
        scale_pos_weight=pos_weight, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'
    ))
])

print(f'Training KNN, Random Forest, and XGBoost (scale_pos_weight={pos_weight:.1f})...')
knn_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)

knn_probs = knn_pipeline.predict_proba(X_test)[:, 1]
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]
xgb_probs = xgb_pipeline.predict_proba(X_test)[:, 1]
print('Model training complete.')

In [ ]:
# 5. Precision-Recall Curves & ROC Comparison
knn_prec, knn_rec, _ = precision_recall_curve(y_test, knn_probs)
rf_prec, rf_rec, _ = precision_recall_curve(y_test, rf_probs)
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_probs)

knn_fpr, knn_tpr, _ = roc_curve(y_test, knn_probs)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve (Gold Standard for imbalanced fraud detection)
axes[0].plot(knn_rec, knn_prec, label=f'KNN (AUPRC = {auc(knn_rec, knn_prec):.3f})', color='#2a9d8f', lw=2.2)
axes[0].plot(rf_rec, rf_prec, label=f'Random Forest (AUPRC = {auc(rf_rec, rf_prec):.3f})', color='#e76f51', lw=2.2)
axes[0].plot(xgb_rec, xgb_prec, label=f'XGBoost (AUPRC = {auc(xgb_rec, xgb_prec):.3f})', color='#1d3557', lw=2.2)
axes[0].set_xlabel('Recall (True Positive Rate)', fontsize=11)
axes[0].set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
axes[0].set_title('Precision-Recall Curve Comparison (Highly Imbalanced Data)', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower left', frameon=True)
axes[0].grid(True, alpha=0.3)

# ROC Curve
axes[1].plot(knn_fpr, knn_tpr, label=f'KNN (AUROC = {roc_auc_score(y_test, knn_probs):.3f})', color='#2a9d8f', lw=2.2)
axes[1].plot(rf_fpr, rf_tpr, label=f'Random Forest (AUROC = {roc_auc_score(y_test, rf_probs):.3f})', color='#e76f51', lw=2.2)
axes[1].plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUROC = {roc_auc_score(y_test, xgb_probs):.3f})', color='#1d3557', lw=2.2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Chance')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
axes[1].set_title('ROC Curves Comparison', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Multi-Model Prediction Scatter Plot Comparison
# Compares the probability confidence assigned by each model vs. transaction amount
test_df = X_test.copy()
test_df['Class'] = y_test.values
test_df['KNN_Prob'] = knn_probs
test_df['RF_Prob'] = rf_probs
test_df['XGB_Prob'] = xgb_probs

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

models_scatter = [
    ('KNN Classifier', 'KNN_Prob', '#2a9d8f'),
    ('Random Forest', 'RF_Prob', '#e76f51'),
    ('XGBoost', 'XGB_Prob', '#1d3557')
]

for i, (m_name, prob_col, base_col) in enumerate(models_scatter):
    # Legitimate transactions
    axes[i].scatter(
        test_df.loc[test_df['Class'] == 0, 'Amount'],
        test_df.loc[test_df['Class'] == 0, prob_col],
        alpha=0.3, s=20, color='#8d99ae', label='Legitimate'
    )
    # Fraud transactions
    axes[i].scatter(
        test_df.loc[test_df['Class'] == 1, 'Amount'],
        test_df.loc[test_df['Class'] == 1, prob_col],
        alpha=0.9, s=55, color='#d90429', edgecolors='black', linewidth=0.7, label='Actual Fraud'
    )
    axes[i].axhline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='Decision Threshold (0.5)')
    axes[i].set_title(f'{m_name}: Fraud Probability vs Amount', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Transaction Amount ($)')
    axes[i].set_xscale('log')
    axes[i].legend(loc='center right', frameon=True, fontsize=9)
    axes[i].grid(True, alpha=0.3)

axes[0].set_ylabel('Predicted Fraud Probability P(Fraud)')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Recurrent Neural Network (LSTM) for Sequential Behavioral Profiling
SEQUENCE_LENGTH = 5

def create_sequences(data, labels, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(labels[i + seq_length])
    return np.array(xs), np.array(ys)

X_sub = StandardScaler().fit_transform(X_train.iloc[:6000].values)
y_sub = y_train.iloc[:6000].values

X_seq, y_seq = create_sequences(X_sub, y_sub, SEQUENCE_LENGTH)
print(f'RNN/LSTM 3D Input Shape: {X_seq.shape}')

rnn_model = Sequential([
    LSTM(32, input_shape=(X_seq.shape[1], X_seq.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(curve='PR', name='auprc')]
)

print('Training Sequential LSTM Behavioral Model...')
history = rnn_model.fit(
    X_seq, y_seq,
    epochs=4,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# Sequence Loss & PR-AUC Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train Loss', color='#1d3557', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e63946', lw=2)
axes[0].set_title('LSTM Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['auprc'], label='Train AUPRC', color='#2a9d8f', lw=2)
axes[1].plot(history.history['val_auprc'], label='Val AUPRC', color='#e76f51', lw=2)
axes[1].set_title('LSTM AUPRC over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()